# Data merge and feature engineering


In [1]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\Uw11\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [3]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw_matrix = pd.read_csv("../data/isw_processed_svd.csv")

In [4]:
df_weather.shape

(844417, 40)

In [5]:
df_war_events.tail()

,datetime_hour,region_id,region_key,alarm_minutes_in_hour,alarm_active
924659,2026-03-16 23:00:00,22,Хмельницька,0.000000,0
924660,2026-03-16 23:00:00,23,Черкаська,0.000000,0
924661,2026-03-16 23:00:00,24,Чернівецька,0.000000,0
924662,2026-03-16 23:00:00,25,Чернігівська,60.000000,1
924663,2026-03-16 23:00:00,26,Київ,17.216667,1


In [6]:
df_isw_matrix.tail()

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
1465,2026-03-12,0.765066,0.254433,0.143732,0.069641,0.144979,-0.166380,0.266070,0.093652,0.047666,...,0.003032,-0.000317,-0.006545,0.011102,0.010367,0.015426,0.005809,0.001704,0.004254,-0.000352
1466,2026-03-13,0.729641,0.246665,0.120514,0.087529,0.148420,-0.158200,0.260993,0.089181,0.039639,...,-0.012584,0.007533,-0.000076,0.013117,0.016895,0.017925,0.022592,0.012608,-0.006074,-0.001135
1467,2026-03-14,0.740226,0.232331,0.208518,0.021025,0.144145,-0.154209,0.256186,0.092777,0.051726,...,-0.016406,0.012282,0.011012,0.000315,0.009572,-0.011610,-0.003205,0.013381,0.005499,0.015785
1468,2026-03-15,0.749144,0.245996,0.145135,0.056091,0.141475,-0.163337,0.259697,0.090983,0.048221,...,-0.004930,0.008439,0.009397,0.014819,0.020686,0.001803,-0.000955,0.014018,-0.008022,0.013834
1469,2026-03-16,0.763329,0.257908,0.159290,0.046706,0.139619,-0.137105,0.237035,0.090248,0.034309,...,-0.024635,-0.002579,-0.007857,-0.023229,-0.002476,0.014476,0.010180,-0.003162,0.001239,0.020061


In [7]:
pd.set_option('display.max_columns', None)

In [8]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [9]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour
193727,2023-01-26 08:00:00,26,-1.3,-3.8,-2.3,-4.7,84.1,0.200,100.0,4.17,20.5,97.0,0.15,-3.3,-3.3,89.36,-4.8,0.0,0.0,0.0,1.0,7.9,0.0,172.6,1028.1,100.0,0.6,0.0,0.0,False,True,False,False,3,8,Kyiv,2023-01-26,07:41:36,16:39:46,Київ,1,60.000000
563645,2024-10-29 16:00:00,7,16.1,3.1,9.8,7.0,84.2,0.124,100.0,4.17,21.6,59.3,0.90,15.0,15.0,66.90,8.9,0.0,0.0,0.0,0.0,13.3,6.8,233.0,1025.1,54.7,133.3,0.5,1.0,False,True,False,False,1,16,Uzhgorod,2024-10-29,07:13:01,17:15:25,Закарпатська,0,0.000000
619345,2025-02-03 09:00:00,3,0.2,-3.4,-1.2,-4.1,81.1,0.596,100.0,4.17,38.5,82.6,0.17,-2.3,-6.8,88.11,-4.0,0.0,0.0,0.0,0.1,23.8,12.6,290.4,1025.0,57.7,31.3,0.1,0.0,False,True,False,False,0,9,Lutsk,2025-02-03,07:51:23,17:14:26,Волинська,0,0.000000
3979,2022-03-02 21:00:00,22,0.7,-1.6,-0.8,-2.2,90.1,0.600,100.0,4.17,29.2,98.4,0.97,-0.3,-3.1,95.02,-1.0,0.0,0.0,0.0,0.1,16.6,7.9,350.9,1019.0,93.8,0.0,0.1,0.0,False,True,False,False,2,21,Khmelnytskyi,2022-03-02,06:53:04,17:56:18,Хмельницька,1,27.750000
284518,2023-07-03 00:00:00,25,26.6,17.5,22.0,14.2,65.2,0.000,0.0,0.00,28.1,60.5,0.50,19.1,19.1,92.76,17.9,0.0,0.0,0.0,0.0,13.0,0.0,257.5,1005.6,60.0,0.0,0.0,0.0,False,True,False,False,0,0,Chernihiv,2023-07-03,04:43:14,21:14:31,Чернігівська,0,0.000000
804741,2026-01-04 02:00:00,26,-0.1,-5.6,-2.7,-6.9,73.9,0.900,100.0,16.67,34.6,45.7,0.53,-2.8,-6.9,73.93,-6.8,0.0,0.0,0.0,6.3,29.9,10.8,230.0,1009.5,75.0,0.0,0.0,0.0,False,True,False,False,6,2,Kyiv,2026-01-04,07:58:11,16:07:49,Київ,0,0.000000
9505,2022-03-12 12:00:00,3,3.9,-6.2,-1.2,-9.2,57.2,0.000,0.0,0.00,32.8,80.0,0.31,1.3,-3.1,47.31,-8.7,0.0,0.0,0.0,0.0,30.2,16.2,310.0,1033.0,88.6,281.0,1.0,3.0,False,True,False,False,5,12,Lutsk,2022-03-12,06:39:33,18:18:19,Волинська,0,0.000000
193137,2023-01-25 08:00:00,11,-1.4,-3.6,-2.0,-3.9,87.0,0.000,0.0,0.00,21.2,95.8,0.12,-1.4,-5.2,88.19,-3.1,0.0,0.0,0.0,0.6,12.6,10.8,10.0,1033.5,100.0,3.6,0.0,0.0,False,True,False,False,2,8,Kropyvnytskyi,2023-01-25,07:29:15,16:37:42,Кіровоградська,0,0.000000
54863,2022-05-30 06:00:00,26,18.6,13.3,15.6,12.2,80.9,3.500,100.0,8.33,43.6,93.7,0.98,13.4,13.4,83.71,10.7,0.0,0.0,0.0,0.0,24.8,3.6,140.0,1014.5,90.0,5.0,0.0,0.0,False,True,False,False,0,6,Kyiv,2022-05-30,04:53:27,20:58:09,Київ,0,0.000000
784586,2025-11-28 14:00:00,19,2.2,0.9,1.6,0.0,89.3,0.000,0.0,0.00,19.8,97.1,0.25,1.2,-1.1,94.40,0.4,0.0,0.0,0.0,0.0,16.6,7.2,70.0,1023.9,100.0,55.7,0.2,1.0,False,True,False,False,4,14,Ternopil,2025-11-28,07:48:43,16:22:21,Тернопільська,0,0.000000


In [10]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,27.7,82.5,0.77,2.1,-0.5,91.76,0.9,0.0,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,True,False,False,3,0,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
24,2022-02-24 01:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,27.7,82.5,0.77,2.1,-0.4,89.80,0.6,0.0,0.0,0.0,0.0,16.6,8.3,289.2,1021.0,97.9,0.0,0.1,0.0,False,True,False,False,3,1,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
48,2022-02-24 02:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,27.7,82.5,0.77,2.0,-0.3,90.44,0.6,0.0,0.0,0.0,0.0,27.7,7.6,287.6,1021.0,98.2,0.0,0.1,0.0,False,True,False,False,3,2,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
72,2022-02-24 03:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,27.7,82.5,0.77,1.9,-0.3,91.75,0.7,0.0,0.0,0.0,0.0,15.1,7.2,288.9,1021.0,98.8,0.0,0.1,0.0,False,True,False,False,3,3,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0
96,2022-02-24 04:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,27.7,82.5,0.77,1.8,-0.2,92.40,0.7,0.0,0.0,0.0,0.0,13.7,6.5,290.4,1021.0,100.0,0.0,0.1,0.0,False,True,False,False,3,4,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0


In [11]:
df_final.tail()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour
844324,2026-03-16 19:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,25.6,15.5,0.92,7.5,7.5,55.25,-0.9,0.0,0.0,0.0,0.0,10.4,3.6,80.0,1019.0,44.3,5.5,0.1,0.0,False,True,False,False,0,19,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,0,0.000000
844347,2026-03-16 20:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,25.6,15.5,0.92,6.9,6.9,59.27,-0.5,0.0,0.0,0.0,0.0,9.4,4.0,79.6,1021.0,59.3,5.5,0.1,0.0,False,True,False,False,0,20,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,0,0.000000
844370,2026-03-16 21:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,25.6,15.5,0.92,6.1,6.1,62.62,-0.5,0.0,0.0,0.0,0.0,10.1,3.6,88.3,1021.0,39.4,5.5,0.1,0.0,False,True,False,False,0,21,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,0,0.000000
844393,2026-03-16 22:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,25.6,15.5,0.92,5.3,5.3,65.23,-0.7,0.0,0.0,0.0,0.0,9.0,2.5,85.6,1021.0,19.4,5.5,0.1,0.0,True,False,False,False,0,22,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,1,25.650000
844416,2026-03-16 23:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,25.6,15.5,0.92,4.7,4.7,66.06,-1.1,0.0,0.0,0.0,0.0,5.8,2.2,63.2,1021.0,10.6,5.5,0.1,0.0,True,False,False,False,0,23,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,1,17.216667


In [12]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 844417 entries, 0 to 844416
Data columns (total 42 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   datetime_hour                  844417 non-null  datetime64[us]
 1   region_id                      844417 non-null  int64         
 2   day_tempmax                    844417 non-null  float64       
 3   day_tempmin                    844417 non-null  float64       
 4   day_temp                       844417 non-null  float64       
 5   day_dew                        844417 non-null  float64       
 6   day_humidity                   844417 non-null  float64       
 7   day_precip                     844417 non-null  float64       
 8   day_precipprob                 844417 non-null  float64       
 9   day_precipcover                844417 non-null  float64       
 10  day_windgust                   844417 non-null  float64       
 11  day_cloudcover  

In [13]:
df_final = df_final.sort_values(["region_id", "datetime_hour"])

df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
df_final["alarm_lag_6"] = df_final.groupby("region_id")["alarm_active"].shift(6)
df_final["alarm_lag_12"] = df_final.groupby("region_id")["alarm_active"].shift(12)
#df_final['alarm_minutes_lag1'] = (df_final.groupby('region_id')['alarm_minutes_in_hour'].shift(1).fillna(0))
#df_final['alarm_minutes_lag3'] = (df_final.groupby('region_id')['alarm_minutes_in_hour'].shift(3).fillna(0))

lag_cols = ["alarm_lag_1","alarm_lag_3","alarm_lag_6","alarm_lag_12"]
df_final[lag_cols] = df_final[lag_cols].fillna(0)

df_final['alarms_in_last_24h'] = (df_final.groupby('region_id')['alarm_active'].transform(lambda x: x.shift(1).rolling(24, min_periods=1).sum()))

df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

hourly_total = df_final.groupby('datetime_hour')['alarm_active'].sum().shift(1)
df_final['total_active_alarms_lag1'] = df_final['datetime_hour'].map(hourly_total)

df_final.sample(10)

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1
2660,2022-02-28 14:00:00,23,3.0,-2.6,0.0,-5.0,69.2,0.000,0.0,0.00,45.4,73.2,0.90,3.0,-2.3,56.54,-4.8,0.0,0.0,0.0,0.0,37.4,27.0,18.5,1029.0,52.8,155.1,0.6,2.0,False,True,False,False,0,14,Cherkasy,2022-02-28,06:36:47,17:32:39,Черкаська,0,0.000000,1.0,0.0,1.0,1.0,14.0,0,0,6.0
566925,2024-11-04 08:00:00,24,8.5,-0.3,4.7,-1.0,67.9,0.000,0.0,0.00,57.6,63.9,0.10,6.7,3.6,42.27,-5.2,0.0,0.0,0.0,0.0,48.2,17.5,302.0,1022.0,98.7,20.1,0.1,0.0,False,True,False,False,0,8,Chernivtsi,2024-11-04,07:07:01,16:52:02,Чернівецька,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,0.0
768669,2025-10-30 18:00:00,18,11.7,5.0,7.9,5.2,84.3,0.100,100.0,4.17,35.6,85.4,0.30,9.1,6.8,76.04,5.1,0.0,0.0,0.0,0.0,25.6,14.8,165.1,1018.0,100.0,0.0,0.0,0.0,False,True,False,False,3,18,Sumy,2025-10-30,06:29:28,16:18:46,Сумська,0,0.000000,1.0,1.0,1.0,1.0,24.0,0,0,5.0
259795,2023-05-21 02:00:00,22,21.2,9.7,15.6,5.2,52.5,0.000,0.0,0.00,38.2,24.4,0.06,11.7,11.7,70.43,6.5,0.0,0.0,0.0,0.0,22.0,11.2,54.3,1023.0,0.0,0.0,0.0,0.0,True,False,False,False,6,2,Khmelnytskyi,2023-05-21,05:21:27,20:56:48,Хмельницька,0,0.000000,0.0,0.0,0.0,0.0,1.0,1,1,1.0
28439,2022-04-14 09:00:00,26,13.9,4.4,8.5,-0.2,58.1,0.000,0.0,0.00,41.8,49.6,0.43,6.2,3.9,68.83,0.9,0.0,0.0,0.0,0.0,33.5,10.8,340.0,1020.1,100.0,310.0,1.1,3.0,False,True,False,False,3,9,Kyiv,2022-04-14,06:06:42,19:50:49,Київ,0,0.000000,0.0,0.0,0.0,0.0,1.0,0,0,2.0
524252,2024-08-22 06:00:00,23,32.9,19.5,26.2,16.5,58.7,0.800,100.0,12.50,34.2,73.8,0.59,19.8,19.8,85.51,17.3,0.0,0.0,0.0,0.0,8.6,6.5,65.4,1007.0,75.7,0.0,0.0,0.0,False,True,False,False,3,6,Cherkasy,2024-08-22,05:53:29,19:54:43,Черкаська,1,26.433333,1.0,0.0,0.0,0.0,9.0,0,1,8.0
111050,2022-09-04 20:00:00,4,16.5,10.9,13.4,7.4,68.1,6.727,100.0,4.17,44.3,72.5,0.25,14.1,14.1,55.79,5.4,0.0,0.0,0.0,0.0,29.2,7.9,9.8,1022.0,3.6,1.0,0.0,0.0,True,False,False,False,6,20,Dnipro,2022-09-04,06:01:02,19:16:08,Дніпропетровська,0,0.000000,1.0,0.0,0.0,0.0,6.0,1,0,4.0
824713,2026-02-09 07:00:00,10,-5.8,-16.3,-10.5,-13.8,77.0,0.000,0.0,0.00,28.4,74.2,0.75,-13.1,-20.2,82.84,-15.4,0.0,0.0,0.0,25.8,24.1,13.7,6.8,1021.0,90.7,0.0,0.0,0.0,False,True,False,False,0,7,Kyiv,2026-02-09,07:20:24,17:04:39,Київська,1,56.683333,1.0,1.0,1.0,0.0,11.0,0,0,10.0
412805,2024-02-10 18:00:00,7,12.3,6.6,9.2,7.1,87.1,7.674,100.0,45.83,36.7,90.8,0.00,9.8,8.2,82.27,6.9,0.0,0.0,0.0,0.0,31.3,11.2,182.0,999.9,86.0,12.3,0.0,0.0,False,True,False,False,5,18,Uzhgorod,2024-02-10,07:48:01,17:42:54,Закарпатська,0,0.000000,0.0,0.0,0.0,0.0,0.0,1,0,5.0
646784,2025-03-23 22:00:00,9,11.1,4.0,6.9,3.0,76.5,0.600,100.0,4.17,40.0,93.6,0.80,6.5,2.9,90.77,5.1,0.0,0.0,0.0,0.0,37.4,19.8,129.4,1016.0,90.8,0.0,0.0,0.0,False,True,False,False,6,22,Ivano-Frankivsk,2025-03-23,06:17:31,18:38:47,Івано-Франківська,0,0.000000,0.0,0.0,0.0,0.0,0.0,1,0,10.0


In [14]:
neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(
    index='datetime_hour',
    columns='region_id',
    values='alarm_active',
    fill_value=0
)

neighbour_alarm_matrix = pd.DataFrame(index=alarms_matrix.index)

for region, neighbours in neighbouring_regions.items():
    valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]
    
    if valid_neighbours:
        neighbour_alarm_matrix[region] = alarms_matrix[valid_neighbours].sum(axis=1)
    else:
        neighbour_alarm_matrix[region] = 0

neighbour_alarm_matrix = neighbour_alarm_matrix.shift(1)

neighbour_alarm_long = (neighbour_alarm_matrix.stack().reset_index())

neighbour_alarm_long.columns = ['datetime_hour','region_id','neighbour_alarms']

df_final = df_final.merge(neighbour_alarm_long,on=['datetime_hour', 'region_id'], how='left')

In [15]:
def hours_since_last_alarm_vectorized(series):
    shifted = series.shift(1).fillna(0)
    alarm_cumsum = shifted.cumsum()
    result = shifted.groupby(alarm_cumsum).cumcount()
    return result

df_final['hours_since_last_alarm'] = (df_final.groupby('region_id')['alarm_active'].transform(hours_since_last_alarm_vectorized))

In [16]:
df_final.sample(20)

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm
522557,2026-01-01 16:00:00,17,-2.5,-16.3,-9.0,-11.4,83.5,0.000,0.0,0.00,42.5,50.0,0.43,-3.3,-8.6,84.74,-5.5,0.0,0.0,0.0,5.4,26.6,15.5,182.0,1008.0,100.0,32.6,0.1,0.0,False,True,False,False,3,16,Rivne,2026-01-01,08:16:31,16:20:52,Рівненська,0,0.000000,1.0,1.0,1.0,1.0,14.0,0,0,10.0,1.0,0
122321,2023-12-07 07:00:00,5,-0.8,-2.4,-1.8,-3.7,87.0,0.000,0.0,0.00,50.0,99.4,0.82,-2.3,-7.8,89.44,-3.8,0.0,0.0,0.0,0.4,37.8,17.6,89.0,1024.0,98.5,0.0,0.0,0.0,False,True,False,False,3,7,Donetsk,2023-12-07,07:04:03,15:35:54,Донецька,1,60.000000,1.0,0.0,0.0,0.0,10.0,0,0,1.0,0.0,0
395953,2023-09-24 22:00:00,14,30.9,12.9,21.8,12.6,61.1,0.000,0.0,0.00,28.1,1.3,0.32,22.3,22.3,51.75,11.9,0.0,0.0,0.0,0.0,21.6,11.5,98.9,1020.0,0.0,0.0,0.0,0.0,True,False,False,False,6,22,Mykolaiv,2023-09-24,06:40:50,18:46:52,Миколаївська,0,0.000000,0.0,1.0,0.0,0.0,4.0,1,0,3.0,1.0,1
350097,2022-07-23 01:00:00,13,32.6,16.3,25.6,15.5,56.6,0.000,0.0,0.00,23.0,48.1,0.80,22.8,22.8,58.72,14.3,0.0,0.0,0.0,0.0,9.4,4.0,132.8,1016.0,31.0,0.0,0.1,0.0,False,True,False,False,5,1,Lviv,2022-07-23,05:41:31,21:18:34,Львівська,0,0.000000,0.0,0.0,0.0,0.0,0.0,1,1,1.0,0.0,151
707379,2022-09-29 11:00:00,23,18.7,9.8,14.4,12.1,86.3,0.000,0.0,0.00,25.2,73.5,0.10,14.4,14.4,79.50,10.9,0.0,0.0,0.0,0.0,14.4,7.9,114.0,1010.0,91.4,356.0,1.3,4.0,False,True,False,False,3,11,Cherkasy,2022-09-29,06:48:09,18:35:15,Черкаська,0,0.000000,0.0,0.0,0.0,0.0,6.0,0,0,6.0,1.0,7
382398,2022-03-09 01:00:00,14,1.3,-1.7,-0.4,-4.6,74.8,0.000,0.0,0.00,38.9,67.9,0.21,-0.4,-0.4,93.64,-1.3,0.0,0.0,0.0,0.0,3.6,2.2,304.3,1014.0,58.0,0.0,0.1,0.0,False,True,False,False,2,1,Mykolaiv,2022-03-09,06:17:08,17:48:54,Миколаївська,0,0.000000,0.0,0.0,0.0,0.0,2.0,0,1,2.0,0.0,9
770283,2025-11-11 18:00:00,24,10.2,5.6,7.7,6.4,92.0,0.000,0.0,0.00,34.6,24.6,0.70,6.9,5.1,86.25,4.8,0.0,0.0,0.0,0.0,18.7,9.5,301.0,1017.0,0.0,0.0,0.0,0.0,True,False,False,False,1,18,Chernivtsi,2025-11-11,07:17:33,16:42:32,Чернівецька,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,8.0,0.0,41
786835,2023-09-11 12:00:00,25,22.3,6.0,14.1,7.8,70.8,0.000,0.0,0.00,17.3,53.2,0.88,22.1,22.1,42.30,8.7,0.0,0.0,0.0,0.0,17.3,7.2,220.0,1021.8,100.0,354.4,1.3,4.0,False,True,False,False,0,12,Chernihiv,2023-09-11,06:22:36,19:19:38,Чернігівська,0,0.000000,0.0,0.0,0.0,1.0,2.0,0,0,4.0,1.0,11
715780,2023-09-14 13:00:00,23,25.2,14.9,19.7,14.0,70.8,3.300,100.0,37.50,26.6,58.3,0.98,25.2,25.2,54.92,15.5,0.0,0.0,0.0,0.0,24.8,12.2,159.5,1015.0,59.0,412.4,1.5,4.0,False,True,False,False,3,13,Cherkasy,2023-09-14,06:25:47,19:08:16,Черкаська,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,3.0,0.0,24
63512,2025-05-03 16:00:00,3,25.1,11.7,17.8,8.4,56.1,1.552,100.0,8.33,51.5,56.1,0.19,24.1,24.1,43.44,10.9,0.0,0.0,0.0,0.0,46.1,20.9,272.1,1008.0,28.7,613.3,2.2,6.0,False,True,False,False,5,16,Lutsk,2025-05-03,05:49:10,20:43:07,Волинська,0,0.000000,0.0,0.0,0.0,0.0,0.0,1,0,3.0,0.0,75


In [17]:
df_isw_matrix.sample(10)

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
1319,2025-10-14,0.747181,0.339812,0.088489,-0.172308,0.001162,-0.144459,0.036787,-0.190518,-0.066613,-0.187446,-0.060652,-0.040023,-0.060026,0.048086,-0.049673,0.004426,0.053710,-0.028178,0.049368,-0.043898,-0.026112,-0.033009,-0.050044,-0.049116,-0.028522,-0.028165,0.015673,-0.024033,-0.018556,0.003075,0.000400,0.042533,0.024088,-0.017701,-0.002570,-0.018683,-0.009724,-0.009970,0.014982,-0.023968,-0.006341,-0.010174,0.005962,-0.046694,0.016984,0.021192,0.036422,0.005376,0.004692,-0.030350,-0.023622,0.008982,0.004246,-0.037845,0.000723,0.010657,0.028145,-0.011919,-0.005896,0.022907,-0.002532,0.028253,-0.002245,0.007506,-0.001162,-0.031922,-0.013847,-0.021181,-0.001041,0.005175,0.007636,0.021677,-0.007297,-0.050674,0.010805,0.008795,-0.028649,-0.000103,-0.002102,0.006754,0.016535,0.061443,0.025110,-0.022785,-0.026930,0.050072,-0.040167,0.032171,-0.009523,0.001851,-0.016568,-0.028217,-0.003518,0.031212,-0.023429,-0.033124,0.008007,-0.012795,0.000577,-0.015566,-0.038264,0.014216,0.019691,-0.027953,0.025071,0.056825,-0.036398,-0.009105,0.015623,0.017508,-0.032763,-0.007155,-0.004493,0.002289,-0.031614,-0.010134,-0.001934,0.005332,-0.048980,0.027041,0.026666,-0.004402,-0.025716,0.006868,-0.021480,0.020490,-0.011712,-0.020468,-0.003296,-0.002507,-0.008206,-0.009635,-0.016837,0.019257,-0.003244,-0.023580,-0.030103,0.004129,-0.007157,0.012231,-0.005970,0.002459,-0.013747,0.030270,0.000095,-0.000582,0.002156,-0.028408,0.012387,0.007780
1313,2025-10-08,0.767828,0.325849,0.028183,-0.123379,0.000494,-0.128035,0.053573,-0.142132,-0.077720,-0.176235,-0.030848,-0.013960,-0.037276,0.070653,-0.088690,0.062491,0.088683,-0.049379,-0.013520,-0.072283,0.017928,0.013510,-0.041348,-0.013895,-0.005288,-0.042731,0.008358,0.022415,-0.031573,0.030920,-0.009761,0.051783,0.003983,0.014097,0.018619,0.025920,-0.003627,-0.008518,0.010372,-0.013575,-0.032885,-0.035461,0.003038,-0.000158,-0.019269,-0.007868,0.052707,-0.015863,-0.013360,0.007603,0.023876,-0.040234,0.025863,-0.024249,-0.013071,-0.058933,-

In [18]:
df_isw_matrix.tail()

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
1465,2026-03-12,0.765066,0.254433,0.143732,0.069641,0.144979,-0.166380,0.266070,0.093652,0.047666,-0.005971,-0.022839,-0.027493,-0.013284,-0.053438,-0.108671,-0.076737,-0.062514,0.074815,0.126928,0.060178,0.064468,-0.073082,0.090115,0.012502,0.025990,-0.021222,0.003271,-0.003000,-0.023399,-0.041246,0.046683,-0.045821,-0.006519,0.023476,-0.031907,0.036555,-0.055942,0.013408,-0.020848,-0.015601,0.019009,0.037012,-0.000524,0.024216,-0.033717,-0.012869,0.025066,-0.040495,0.038080,0.030960,-0.043235,0.010624,-0.045864,0.009136,-0.033719,-0.005913,0.001567,-0.006515,0.003483,-0.001836,0.013089,-0.016532,-0.003341,0.013519,0.019499,-0.024948,0.010420,-0.018005,-6.777517e-03,-0.005597,-0.002244,0.009911,-0.008649,-0.011712,-0.015743,0.015824,-0.031023,0.025343,0.017240,0.008802,-0.027629,0.014344,-0.026334,-0.011674,-0.016208,0.016379,-0.011220,0.001471,-0.013165,-0.009175,-0.037000,-0.000638,-0.014463,-0.014415,0.006220,-0.004982,-0.000845,0.003552,-0.007706,-0.013666,0.010243,0.035163,0.014926,0.002759,0.011273,0.004127,0.014570,0.011445,0.007651,-0.011859,-0.009574,-0.007425,-0.002650,0.011232,0.009197,0.015446,-0.010712,-0.009066,-0.003153,-0.016597,0.010077,-0.002519,0.011182,-0.018032,-0.004637,-0.013795,0.023275,-0.012627,0.011217,0.008873,0.001366,0.014713,-0.001550,0.011899,0.015634,-0.008881,-0.000234,-0.025042,0.017235,-0.029331,0.003032,-0.000317,-0.006545,0.011102,0.010367,0.015426,0.005809,0.001704,0.004254,-0.000352
1466,2026-03-13,0.729641,0.246665,0.120514,0.087529,0.148420,-0.158200,0.260993,0.089181,0.039639,-0.021073,-0.001887,-0.029413,-0.016107,-0.043593,-0.123176,-0.029581,-0.045643,0.043537,0.109786,0.033566,0.032703,-0.014846,0.109468,0.066838,0.009854,-0.021243,-0.052359,0.047748,0.004632,-0.003190,0.048814,-0.072675,-0.020785,0.014451,-0.033611,0.044921,-0.017667,0.014083,-0.021032,-0.034673,0.004057,-0.012496,-0.022244,0.010415,-0.087610,-0.050536,0.071161,-0.035159,0.020357,0.033910,-0.026163,0.015623,-0.024740,-0.048221,-0.013629,-0.007749,0.026537

In [19]:
#df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)    ТУТ ЗСУВ ІСВ НА + 1 ДЕНЬ
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [20]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime']).dt.date
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime']).dt.date
df_isw_matrix.fillna(0, inplace=True)
df_isw_matrix.isna().sum()

day_datetime     0
isw_topic_0      0
isw_topic_1      0
isw_topic_2      0
isw_topic_3      0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 151, dtype: int64

In [21]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [22]:
df_final.head(10)

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,27.7,82.5,0.77,2.1,-0.5,91.76,0.9,0.0,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,True,False,False,3,0,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.0,0.0,0.0,0.0,0.0,NaN,0,1,NaN,NaN,0,0.648733,-0.078386,0.180873,0.059880,0.070008,-0.042071,-0.006712,0.006174,-0.014324,0.004678,-0.034546,0.026839,0.051262,0.052505,0.069154,-0.065115,0.063161,-0.13259,-0.148380,0.167924,-0.069896,0.082404,-0.099159,-0.135181,0.040913,0.067235,-0.000759,0.058867,-0.037716,0.038137,0.020267,0.064787,-0.092394,-0.073454,-0.044226,0.015863,-0.045746,0.052133,-0.060692,0.157277,0.045770,0.153091,0.048066,-0.040708,0.029668,-0.018682,0.045509,0.093437,-0.040995,0.078446,-0.038387,-0.054725,-0.011545,-0.097217,0.008896,-0.047726,0.027129,0.028893,0.040973,0.061139,0.005002,-0.081452,-0.032886,0.071956,0.016690,0.039795,0.046933,-0.017759,0.000617,0.055723,0.001567,-0.065267,0.039934,-0.036891,0.055054,0.077850,-0.027169,0.051738,-0.012014,0.012023,-0.088707,0.027350,0.043625,0.012068,-0.010631,0.054585,-0.036056,-0.057756,-0.029479,0.036439,0.021386,-0.003892,-0.004663,-0.036183,0.014969,-0.000180,0.036404,-0.045506,-0.041311,0.045422,0.034035,0.045637,0.034013,0.002362,-0

In [23]:
df_final.isna().sum()

datetime_hour       0
region_id           0
day_tempmax         0
day_tempmin         0
day_temp            0
                 ... 
isw_topic_145    7416
isw_topic_146    7416
isw_topic_147    7416
isw_topic_148    7416
isw_topic_149    7416
Length: 202, dtype: int64

In [24]:
df_final.tail()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
844988,2026-03-16 19:00:00,26,11.1,0.1,5.6,-1.8,60.9,0.0,0.0,0.0,25.6,15.5,0.92,7.5,7.5,55.25,-0.9,0.0,0.0,0.0,0.0,10.4,3.6,80.0,1019.0,44.3,5.5,0.1,0.0,False,True,False,False,0,19,Kyiv,2026-03-16,06:09:51,18:04:17,Київ,0,0.000000,0.0,0.0,0.0,0.0,3.0,0,0,8.0,1.0,7,0.763329,0.257908,0.15929,0.046706,0.139619,-0.137105,0.237035,0.090248,0.034309,-0.016405,-0.007182,-0.020786,-0.015444,-0.040635,-0.087929,-0.065333,-0.031703,0.065081,0.107947,0.095459,0.061175,-0.072911,0.100152,-0.004679,0.019504,-0.024268,-0.012686,-0.003419,0.009949,-0.016268,0.050409,-0.067728,-0.001722,0.007696,-0.052572,0.027906,-0.043968,0.009228,-0.022924,-0.026173,0.015389,0.049728,-0.011892,0.035544,-0.040425,-0.015477,0.027806,-0.048095,0.006576,0.033667,-0.053193,0.002926,-0.024577,0.01843,-0.025015,-0.002874,0.015761,-0.006361,0.015737,0.049314,-0.000153,-0.011353,-0.005248,0.02792,0.010209,-0.018895,0.028934,-0.009222,0.014866,0.00056,-0.045952,-0.00898,-0.018907,0.001456,-0.022851,-0.006765,-0.04409,0.009906,0.031205,0.031027,-0.019364,0.001277,-0.009505,0.006802,-0.021599,0.006317,-0.014554,-0.030289,0.010772,-0.024763,-0.028286,0.003788,0.010264,-0.027466,-0.008981,-0.01088,-0.002958,-0.020453,-0.001468,0.024672,0.012768,0.021246,0.022114,-0.01389

In [25]:
df_final.fillna(0, inplace=True)

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24 00:00:00,2,5.0,0.7,2.8,-0.3,80.5,0.3,100.0,4.17,27.7,82.5,0.77,2.1,-0.5,91.76,0.9,0.0,0.0,0.0,0.0,16.6,8.6,279.6,1021.0,91.1,0.0,0.1,0.0,False,True,False,False,3,0,Vinnytsia,2022-02-24,06:58:49,17:40:52,Вінницька,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,1,0.0,0.0,0,0.648733,-0.078386,0.180873,0.059880,0.070008,-0.042071,-0.006712,0.006174,-0.014324,0.004678,-0.034546,0.026839,0.051262,0.052505,0.069154,-0.065115,0.063161,-0.132590,-0.148380,0.167924,-0.069896,0.082404,-0.099159,-0.135181,0.040913,0.067235,-0.000759,0.058867,-0.037716,0.038137,0.020267,0.064787,-0.092394,-0.073454,-0.044226,0.015863,-0.045746,0.052133,-0.060692,0.157277,0.045770,0.153091,0.048066,-0.040708,0.029668,-0.018682,0.045509,0.093437,-0.040995,0.078446,-0.038387,-0.054725,-0.011545,-0.097217,0.008896,-0.047726,0.027129,0.028893,0.040973,0.061139,0.005002,-0.081452,-0.032886,0.071956,0.016690,0.039795,0.046933,-0.017759,0.000617,0.055723,0.001567,-0.065267,0.039934,-0.036891,0.055054,0.077850,-0.027169,0.051738,-0.012014,0.012023,-0.088707,0.027350,0.043625,0.012068,-0.010631,0.054585,-0.036056,-0.057756,-0.029479,0.036439,0.021386,-0.003892,-0.004663,-0.036183,0.014969,-0.000180,0.036404,-0.045506,-0.041311,0.045422,0.034035,0.045637,0.034013,0.002

In [26]:
df_final.isna().sum()

datetime_hour    0
region_id        0
day_tempmax      0
day_tempmin      0
day_temp         0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 202, dtype: int64

### Feature engineering for isw topics 

In [27]:
topic_cols = [c for c in df_final.columns if "isw_topic_" in c]
df_isw_abs = df_final[topic_cols].abs()

df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
df_final['isw_topic_max'] = df_isw_abs.max(axis=1)
df_final['isw_topic_mean'] = df_isw_abs.mean(axis=1)

df_final['isw_velocity_24h'] = (
    df_final.groupby('region_id')['isw_total_intensity']
    .diff(24)
    .fillna(0)
)

df_final['isw_intensity_ema'] = (df_final.groupby('region_id')['isw_total_intensity'].transform(lambda x: x.shift(1).ewm(span=24).mean()))

df_final['isw_topic_entropy'] = -(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) * np.log(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

isw_cols = ['isw_velocity_24h', 'isw_intensity_ema', 'isw_topic_entropy']
df_final[isw_cols] = df_final[isw_cols].fillna(0)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_29284\1269306822.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_29284\1269306822.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_29284\1269306822.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

In [28]:
df_final.sample(10)

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy
78032,2022-12-05 17:00:00,4,-3.4,-9.0,-6.8,-12.6,63.9,0.000,0.0,0.00,37.8,5.1,0.37,-4.7,-10.0,57.58,-11.8,0.0,0.0,0.0,0.0,34.2,14.0,81.5,1035.0,0.0,0.0,0.1,0.0,True,False,False,False,0,17,Dnipro,2022-12-05,07:15:10,15:45:38,Дніпропетровська,0,0.000000,1.0,1.0,0.0,0.0,9.0,0,0,24.0,7.0,0,0.602673,-0.107887,-0.062862,-0.046238,0.104967,0.009789,-0.122709,0.170461,-0.134839,-0.028030,-0.071228,-0.011209,0.010958,-0.000430,-0.104796,0.012517,-0.074765,0.035123,-0.051429,0.129223,0.022825,0.024960,-0.016663,0.103264,-0.038798,0.055985,-0.077021,0.133471,-0.028440,-0.111060,0.046021,0.081770,0.025553,0.103138,-0.033967,0.095586,-0.079968,-0.039622,-0.100816,0.075453,-0.019357,0.017407,-0.023932,-0.046530,0.079786,0.039166,0.044172,-0.007309,-0.033271,-0.003359,0.070337,-0.026430,0.110314,0.029114,-0.025437,0.010074,-0.093694,0.006347,0.013641,0.033421,-0.014271,-0.053559,0.030952,0.059691,0.056225,-0.020292,-0.016448,0.042065,-0.002877,-0.004276,0.094276,-0.010740,-0.001885,0.001512,0.038723,-0.005771,0.016822,0.001133,-0.096306,-0.069449,-0.046535,-0.082070,-0.042402,-0.049618,-0.004135,-0.021530,-0.004639,0.009675,-0.0

### TELEGRAM MERGE

In [29]:
df_tg_matrix = pd.read_csv("../data/telegram_processed_svd.csv")

In [30]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,tg_topic_111,tg_topic_112,tg_topic_113,tg_topic_114,tg_topic_115,tg_topic_116,tg_topic_117,tg_topic_118,tg_topic_119,tg_topic_120,tg_topic_121,tg_topic_122,tg_topic_123,tg_topic_124,tg_topic_125,tg_topic_126,tg_topic_127,tg_topic_128,tg_topic_129,tg_topic_130,tg_topic_131,tg_topic_132,tg_topic_133,tg_topic_134,tg_topic_135,tg_topic_136,tg_topic_137,tg_topic_138,tg_topic_139,tg_topic_140,tg_topic_141,tg_topic_142,tg_topic_143,tg_topic_144,tg_topic_145,tg_topic_146,tg_topic_147,tg_topic_148,tg_topic_149,tg_topic_150,tg_topic_151,tg_topic_152,tg_topic_153,tg_topic_154,tg_topic_155,tg_topic_156,tg_topic_157,tg_topic_158,tg_topic_159,tg_topic_160,tg_topic_161,tg_topic_162,tg_topic_163,tg_topic_164,tg_topic_165,tg_topic_166,tg_topic_167,tg_topic_168,tg_topic_169,tg_topic_170,tg_topic_171,tg_topic_172,tg_topic_173,tg_topic_174,tg_topic_175,tg_topic_176,tg_topic_177,tg_topic_178,tg_topic_179,tg_topic_180,tg_topic_181,tg_topic_182,tg_topic_183,tg_topic_184,tg_topic_185,tg_topic_186,tg_topic_187,tg_topic_188,tg_topic_189,tg_topic_190,tg_topic_191,tg_topic_192,tg_topic_193,tg_topic_194,tg_topic_195,tg_topic_196,tg_topic_197,tg_topic_198,tg_topic_199,tg_topic_200,tg_topic_201,tg_topic_202,tg_topic_203,tg_topic_204,tg_topic_205,tg_topic_206,tg_topic_207,tg_topic_208,tg_topic_209,tg_topic_210,tg_topic_211,tg_topic_212,tg_topic_213,tg_topic_214,tg_topic_215,tg_topic_216,tg_topic_217,tg_topic_218,tg_topic_219,tg_topic_220,tg_topic_221,tg_topic_222,tg_topic_223,tg_topic_224,tg_topic_225,tg_topic_226,tg_topic_227,tg_topic_228,tg_topic_229,tg_topic_230,tg_topic_231,tg_topic_232,tg_topic_233,tg_topic_234,tg_topic_235,tg_topic_236,tg_topic_237,tg_topic_238,tg_topic_239,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2026-03-17 01:31:21,kpszsu,0.138465,-0.060347,0.284671,0.035601,0.116456,-0.045463,-0.021404,0.031473,0.003471,0.014402,0.286607,0.194596,-0.210486,0.069280,-0.111417,0.052725,0.069214,0.119048,-0.077877,0.065132,-0.077492,-0.005224,-0.019193,0.070873,-0.107137,-0.076956,-0.067567,0.098249,0.002130,0.136716,0.044741,-0.024770,0.051995,-0.083287,0.067505,0.114771,0.033290,0.000556,0.255589,-0.002123,-0.049321,0.032166,-0.022657,-0.121028,0.038802,0.038982,0.039210,0.119676,0.006456,-0.083387,-0.005215,0.086310,-0.061551,-0.063818,0.027737,-0.027433,0.029990,-0.021397,-0.062931,-0.081544,-0.037276,0.086436,0.086050,0.001841,0.071080,-0.013168,0.000673,-0.012356,-0.001683,0.025808,-0.025291,-0.025418,0.053668,0.007258,-0.024991,0.030166,0.059870,-0.083016,-0.066038,-0.054990,0.013588,0.063259,0.006870,-0.141285,0.103932,-0.012589,0.006

In [31]:
df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")

C:\Users\Uw11\AppData\Local\Temp\ipykernel_29284\2851151304.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")


In [32]:
topic_cols = [c for c in df_tg_matrix.columns if "tg_topic_" in c]

tg_hourly = (
    df_tg_matrix.groupby("datetime_hour")[topic_cols]
    .mean()
    .sort_index()
    .reset_index()
)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_29284\2188712141.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


In [33]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,tg_topic_95,tg_topic_96,tg_topic_97,tg_topic_98,tg_topic_99,tg_topic_100,tg_topic_101,tg_topic_102,tg_topic_103,tg_topic_104,tg_topic_105,tg_topic_106,tg_topic_107,tg_topic_108,tg_topic_109,tg_topic_110,tg_topic_111,tg_topic_112,tg_topic_113,tg_topic_114,tg_topic_115,tg_topic_116,tg_topic_117,tg_topic_118,tg_topic_119,tg_topic_120,tg_topic_121,tg_topic_122,tg_topic_123,tg_topic_124,tg_topic_125,tg_topic_126,tg_topic_127,tg_topic_128,tg_topic_129,tg_topic_130,tg_topic_131,tg_topic_132,tg_topic_133,tg_topic_134,tg_topic_135,tg_topic_136,tg_topic_137,tg_topic_138,tg_topic_139,tg_topic_140,tg_topic_141,tg_topic_142,tg_topic_143,tg_topic_144,tg_topic_145,tg_topic_146,tg_topic_147,tg_topic_148,tg_topic_149,tg_topic_150,tg_topic_151,tg_topic_152,tg_topic_153,tg_topic_154,tg_topic_155,tg_topic_156,tg_topic_157,tg_topic_158,tg_topic_159,tg_topic_160,tg_topic_161,tg_topic_162,tg_topic_163,tg_topic_164,tg_topic_165,tg_topic_166,tg_topic_167,tg_topic_168,tg_topic_169,tg_topic_170,tg_topic_171,tg_topic_172,tg_topic_173,tg_topic_174,tg_topic_175,tg_topic_176,tg_topic_177,tg_topic_178,tg_topic_179,tg_topic_180,tg_topic_181,tg_topic_182,tg_topic_183,tg_topic_184,tg_topic_185,tg_topic_186,tg_topic_187,tg_topic_188,tg_topic_189,tg_topic_190,tg_topic_191,tg_topic_192,tg_topic_193,tg_topic_194,tg_topic_195,tg_topic_196,tg_topic_197,tg_topic_198,tg_topic_199,tg_topic_200,tg_topic_201,tg_topic_202,tg_topic_203,tg_topic_204,tg_topic_205,tg_topic_206,tg_topic_207,tg_topic_208,tg_topic_209,tg_topic_210,tg_topic_211,tg_topic_212,tg_topic_213,tg_topic_214,tg_topic_215,tg_topic_216,tg_topic_217,tg_topic_218,tg_topic_219,tg_topic_220,tg_topic_221,tg_topic_222,tg_topic_223,tg_topic_224,tg_topic_225,tg_topic_226,tg_topic_227,tg_topic_228,tg_topic_229,tg_topic_230,tg_topic_231,tg_topic_232,tg_topic_233,tg_topic_234,tg_topic_235,tg_topic_236,tg_topic_237,tg_topic_238,tg_topic_239,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249,datetime_hour
0,2026-03-17 01:31:21,kpszsu,0.138465,-0.060347,0.284671,0.035601,0.116456,-0.045463,-0.021404,0.031473,0.003471,0.014402,0.286607,0.194596,-0.210486,0.069280,-0.111417,0.052725,0.069214,0.119048,-0.077877,0.065132,-0.077492,-0.005224,-0.019193,0.070873,-0.107137,-0.076956,-0.067567,0.098249,0.002130,0.136716,0.044741,-0.024770,0.051995,-0.083287,0.067505,0.114771,0.033290,0.000556,0.255589,-0.002123,-0.049321,0.032166,-0.022657,-0.121028,0.038802,0.038982,0.039210,0.119676,0.006456,-0.083387,-0.005215,0.086310,-0.061551,-0.063818,0.027737,-0.027433,0.029990,-0.021397,-0.062931,-0.081544,-0.037276,0.086436,0.086050,0.001841,0.071080,-0.013168,0.000673,-0.012356,-0.001683,0.025808,-0.025291,-0.025418,0.053668,0.007258,-0.024991,0.030166,0.059870,-0.083016,-0.066038,-0.054990,0.013588,0.063259,0.006870,-0.141285,0.103932,-

In [34]:
#tg_hourly["datetime_hour"] = tg_hourly["datetime_hour"] + pd.Timedelta(hours=1)        ТУТ ЗСУВ ТГ НА 1 ГОДИНУ 

In [35]:
df_final = df_final.merge(tg_hourly, on="datetime_hour", how="left")

In [36]:
df_final.sample(20)

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,t

In [37]:
df_final.shape

(844993, 459)

In [38]:
df_final = df_final.sort_values(["datetime_hour"])

In [39]:
with pd.option_context('display.max_rows', None):
    print(df_final.isnull().sum())

datetime_hour                         0
region_id                             0
day_tempmax                           0
day_tempmin                           0
day_temp                              0
day_dew                               0
day_humidity                          0
day_precip                            0
day_precipprob                        0
day_precipcover                       0
day_windgust                          0
day_cloudcover                        0
day_moonphase                         0
hour_temp                             0
hour_feelslike                        0
hour_humidity                         0
hour_dew                              0
hour_precip                           0
hour_precipprob                       0
hour_snow                             0
hour_snowdepth                        0
hour_windgust                         0
hour_windspeed                        0
hour_winddir                          0
hour_pressure                         0


In [40]:
df_final = df_final.fillna(0)

In [41]:
tg_cols = [c for c in df_final.columns if 'tg_topic_' in c]
df_tg_abs = df_final[tg_cols].abs()

df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
df_final['tg_topic_max'] = df_tg_abs.max(axis=1)
df_final['tg_topic_entropy'] = -(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) * np.log(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_29284\409095542.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_29284\409095542.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_29284\409095542.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performanc

In [42]:
df_final['tg_velocity_3h'] = (df_final.groupby('region_id')['tg_total_intensity'].diff(3).fillna(0))
df_final['tg_intensity_ema_6h'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: x.ewm(span=6).mean()))
df_final['tg_intensity_zscore'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: (x - x.rolling(24, min_periods=1).mean()) / (x.rolling(24, min_periods=1).std() + 1e-9)))

C:\Users\Uw11\AppData\Local\Temp\ipykernel_29284\3857173777.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_velocity_3h'] = (df_final.groupby('region_id')['tg_total_intensity'].diff(3).fillna(0))
C:\Users\Uw11\AppData\Local\Temp\ipykernel_29284\3857173777.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_intensity_ema_6h'] = (df_final.groupby('region_id')['tg_total_intensity'].transform(lambda x: x.ewm(span=6).mean()))
C:\Users\Uw11\AppData\Local\Temp\ipykernel_29284\3857173777.py:3: PerformanceWa

In [43]:
df_final.sample(20)

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,t

# ТУТ БУДЕ EDA

## Weather, ISW, Telegram shift for further models trainings

In [44]:
df_to_train = df_final.copy()
df_to_train = df_to_train.sort_values(['region_id', 'datetime_hour'])

In [45]:
isw_cols = [c for c in df_to_train.columns if 'isw_' in c]
df_to_train[isw_cols] = df_to_train.groupby('region_id')[isw_cols].shift(24).fillna(0)

In [46]:
df_to_train.head()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,t

In [47]:
tg_cols = [c for c in df_to_train.columns if 'tg_' in c]
df_to_train[tg_cols] = df_to_train.groupby('region_id')[tg_cols].shift(1).fillna(0)

In [48]:
df_to_train.head(5)

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,t

In [49]:
hour_weather_cols = [c for c in df_to_train.columns if c.startswith('hour_')]
for col in hour_weather_cols:
    if df_to_train[col].dtype == bool:
        df_to_train[col] = df_to_train.groupby('region_id')[col].shift(1).fillna(False)
    else:
        df_to_train[col] = df_to_train.groupby('region_id')[col].shift(1).fillna(0)

In [50]:
day_weather_cols = [c for c in df_to_train.columns if (c.startswith('day_') and c not in ['day_datetime', 'day_sunrise', 'day_sunset', 'day_moonphase', 'day_of_week'])]
df_to_train[day_weather_cols] = df_to_train.groupby('region_id')[day_weather_cols].shift(24).fillna(0)

In [51]:
df_to_train.isna().sum()

datetime_hour          0
region_id              0
day_tempmax            0
day_tempmin            0
day_temp               0
                      ..
tg_topic_max           0
tg_topic_entropy       0
tg_velocity_3h         0
tg_intensity_ema_6h    0
tg_intensity_zscore    0
Length: 466, dtype: int64

In [52]:
df_to_train['alarm_minutes_in_hour'] = (df_to_train.groupby('region_id')['alarm_minutes_in_hour'].shift(1).fillna(0))

In [53]:
df_to_train.head()

,datetime_hour,region_id,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,day_of_week,hour,city_name,day_datetime,day_sunrise,day_sunset,region_key,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,isw_topic_9,isw_topic_10,isw_topic_11,isw_topic_12,isw_topic_13,isw_topic_14,isw_topic_15,isw_topic_16,isw_topic_17,isw_topic_18,isw_topic_19,isw_topic_20,isw_topic_21,isw_topic_22,isw_topic_23,isw_topic_24,isw_topic_25,isw_topic_26,isw_topic_27,isw_topic_28,isw_topic_29,isw_topic_30,isw_topic_31,isw_topic_32,isw_topic_33,isw_topic_34,isw_topic_35,isw_topic_36,isw_topic_37,isw_topic_38,isw_topic_39,isw_topic_40,isw_topic_41,isw_topic_42,isw_topic_43,isw_topic_44,isw_topic_45,isw_topic_46,isw_topic_47,isw_topic_48,isw_topic_49,isw_topic_50,isw_topic_51,isw_topic_52,isw_topic_53,isw_topic_54,isw_topic_55,isw_topic_56,isw_topic_57,isw_topic_58,isw_topic_59,isw_topic_60,isw_topic_61,isw_topic_62,isw_topic_63,isw_topic_64,isw_topic_65,isw_topic_66,isw_topic_67,isw_topic_68,isw_topic_69,isw_topic_70,isw_topic_71,isw_topic_72,isw_topic_73,isw_topic_74,isw_topic_75,isw_topic_76,isw_topic_77,isw_topic_78,isw_topic_79,isw_topic_80,isw_topic_81,isw_topic_82,isw_topic_83,isw_topic_84,isw_topic_85,isw_topic_86,isw_topic_87,isw_topic_88,isw_topic_89,isw_topic_90,isw_topic_91,isw_topic_92,isw_topic_93,isw_topic_94,isw_topic_95,isw_topic_96,isw_topic_97,isw_topic_98,isw_topic_99,isw_topic_100,isw_topic_101,isw_topic_102,isw_topic_103,isw_topic_104,isw_topic_105,isw_topic_106,isw_topic_107,isw_topic_108,isw_topic_109,isw_topic_110,isw_topic_111,isw_topic_112,isw_topic_113,isw_topic_114,isw_topic_115,isw_topic_116,isw_topic_117,isw_topic_118,isw_topic_119,isw_topic_120,isw_topic_121,isw_topic_122,isw_topic_123,isw_topic_124,isw_topic_125,isw_topic_126,isw_topic_127,isw_topic_128,isw_topic_129,isw_topic_130,isw_topic_131,isw_topic_132,isw_topic_133,isw_topic_134,isw_topic_135,isw_topic_136,isw_topic_137,isw_topic_138,isw_topic_139,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,tg_topic_8,tg_topic_9,tg_topic_10,tg_topic_11,tg_topic_12,tg_topic_13,tg_topic_14,tg_topic_15,tg_topic_16,tg_topic_17,tg_topic_18,tg_topic_19,tg_topic_20,tg_topic_21,tg_topic_22,tg_topic_23,tg_topic_24,tg_topic_25,tg_topic_26,tg_topic_27,tg_topic_28,tg_topic_29,tg_topic_30,tg_topic_31,tg_topic_32,tg_topic_33,tg_topic_34,tg_topic_35,tg_topic_36,tg_topic_37,tg_topic_38,tg_topic_39,tg_topic_40,tg_topic_41,tg_topic_42,tg_topic_43,tg_topic_44,tg_topic_45,tg_topic_46,tg_topic_47,tg_topic_48,tg_topic_49,tg_topic_50,tg_topic_51,tg_topic_52,tg_topic_53,tg_topic_54,tg_topic_55,tg_topic_56,tg_topic_57,tg_topic_58,tg_topic_59,tg_topic_60,tg_topic_61,tg_topic_62,tg_topic_63,tg_topic_64,tg_topic_65,tg_topic_66,tg_topic_67,tg_topic_68,tg_topic_69,tg_topic_70,tg_topic_71,tg_topic_72,tg_topic_73,tg_topic_74,tg_topic_75,tg_topic_76,tg_topic_77,tg_topic_78,tg_topic_79,tg_topic_80,tg_topic_81,tg_topic_82,tg_topic_83,tg_topic_84,tg_topic_85,tg_topic_86,tg_topic_87,tg_topic_88,tg_topic_89,tg_topic_90,tg_topic_91,tg_topic_92,tg_topic_93,tg_topic_94,t

In [55]:
print(f"Final dataset shape: {df_to_train.shape}")
print(f"Date range: {df_to_train['datetime_hour'].min()} → {df_to_train['datetime_hour'].max()}")
print(f"Regions: {df_to_train['region_id'].nunique()}")
print(f"Target balance: {df_to_train['alarm_active'].mean():.2%} positive")
print(f"NaN count: {df_to_train.isna().sum().sum()}")

Final dataset shape: (844993, 466)
Date range: 2022-02-24 00:00:00 → 2026-03-16 23:00:00
Regions: 24
Target balance: 20.64% positive
NaN count: 0


In [54]:
df_to_train.to_parquet("final_merged_dataset.parquet", index = False, engine = "pyarrow")